# G1 Academy Bonus - Task 11: Dex3 gradual gripping (baseline, close, and open)

## Introduction
A Dex3 `HandCmd` message carries one motor command per hand joint; every commanded joint needs control mode, target position, and conservative `dq`/`tau`/`kp`/`kd`. This task measures an unloaded tactile noise baseline (per `academy/todo.txt`: thresholds must be calibrated per installation, never hardcoded), then implements both directions of gradual motion - `gradual_close` (stopping early on tactile contact) and `gradual_open` (a graceful, symmetric release, filling the gap `academy/todo.txt` flags: *"Add a direct graceful hand-open implementation alongside the completed gradual close example."*)

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

import sys
if ".." not in sys.path:
    sys.path.append("..")
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Tactile subscriber + hand publisher
`HandState_` exposes `motor_state` (per-joint q/dq/tau) and `press_sensor_state` (tactile pressure arrays). `HAND_OPEN`/`HAND_CLOSED` come from `util.py`'s documented target-array mapping - not a hidden hand client.

In [ ]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__HandCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import HandCmd_, HandState_
import sys
sys.path.append("..")
from util import HAND_JOINT_NAMES, HAND_OPEN, HAND_CLOSED

hand_state = None
def _on_hand_state(msg):
    global hand_state
    hand_state = msg

hand_pub = ChannelPublisher("rt/dex3/right/cmd", HandCmd_); hand_pub.Init()
hand_state_sub = ChannelSubscriber("rt/dex3/right/state", HandState_); hand_state_sub.Init(_on_hand_state, 20)

_last_hand_targets = {}
def write_hand(targets, side="right", kp=0.8, kd=0.05, tau=0.02):
    msg = unitree_hg_msg_dds__HandCmd_()
    for i, q in enumerate(targets):
        cmd = msg.motor_cmd[i]
        cmd.mode = (i & 0x0F) | (1 << 4); cmd.q = float(q); cmd.dq = 0.0; cmd.tau = tau; cmd.kp = kp; cmd.kd = kd
    hand_pub.Write(msg)
    _last_hand_targets[side] = list(targets)

def tactile_samples():
    if hand_state is None:
        return []
    return [float(v) for sensor in hand_state.press_sensor_state for v in sensor.pressure]

print(tactile_samples())

## Task 2 - `measure_baseline()`: unloaded tactile noise floor
Record no-object noise before picking any grasp-contact threshold - approved thresholds are empirical per installation and per object class, not a constant to copy from another robot.

In [ ]:
def measure_baseline(duration_s=2.0):
    samples = []
    deadline = time.time() + duration_s
    while time.time() < deadline:
        samples.extend(tactile_samples())
        time.sleep(0.02)
    if not samples:
        raise RuntimeError("No tactile samples received; is the hand publishing state?")
    return {"max": max(samples), "mean": sum(samples) / len(samples), "n": len(samples)}

baseline = measure_baseline()
print(baseline)  # pick a threshold comfortably above baseline['max']

## Task 3 - `gradual_close(threshold, ...)`
Interpolate from `HAND_OPEN` to `HAND_CLOSED` in small steps, stopping the moment any tactile sensor crosses the calibrated `threshold` (from a supervised safe grasp, well above the measured baseline).

In [ ]:
def gradual_close(threshold, side="right", steps=40, delay_s=0.05):
    start = _last_hand_targets.get(side, HAND_OPEN[side])
    for step in range(1, steps + 1):
        a = step / steps
        frame = [x + (y - x) * a for x, y in zip(start, HAND_CLOSED[side])]
        write_hand(frame, side=side)
        time.sleep(delay_s)
        samples = tactile_samples()
        if samples and max(samples) >= threshold:
            return {"contact": True, "step": step, "side": side}
    return {"contact": False, "step": steps, "side": side}

gradual_close(threshold=103180)

## Task 4 - `gradual_open(...)`: the graceful release
Symmetric to `gradual_close`, ramping smoothly from whatever pose the hand is currently commanded to (which may be a partial grasp, not necessarily fully closed) back to `HAND_OPEN`. There is no tactile early-stop here - opening should just be gentle rather than snapping open, which can drop a held object suddenly or shock the fingers against their end stops.

In [ ]:
def gradual_open(side="right", steps=40, delay_s=0.05):
    start = _last_hand_targets.get(side, HAND_CLOSED[side])
    for step in range(1, steps + 1):
        a = step / steps
        frame = [x + (y - x) * a for x, y in zip(start, HAND_OPEN[side])]
        write_hand(frame, side=side)
        time.sleep(delay_s)
    return {"opened": True, "side": side}

gradual_open()

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.